# pandas Series — the one-dimensional building block

A `Series` is a 1-D array **plus an index**. Nearly every DataFrame operation returns one
(`df["col"]`, `df.mean()`, a `groupby().sum()` on one column), so being fluent with Series
means being fluent with pandas.

**What's in here**
- anatomy: values, index, name, dtype
- creating Series from lists, dicts, scalars, DataFrame columns
- selecting: `[]`, `.loc`, `.iloc`, slices, masks, `.at/.iat`
- index alignment in arithmetic (the thing that silently produces NaN)
- missing values, `where` / `mask`
- vectorised methods: stats, `value_counts`, ranks, cumulative ops, shifts, rolling
- `map` / `apply` / `replace` / `astype`, `.str`, `.dt`, `.cat` accessors
- index manipulation: `reindex`, `rename`, `reset_index`, duplicates in the index
- combining Series: `concat`, `combine_first`, `update`, `to_frame`
- MultiIndex Series from a two-key groupby
- conversions and a pitfall catalogue

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
cons = df.set_index("time")["consumption_mwh"]   # a Series indexed by time
type(cons), cons.shape, cons.dtype

(pandas.core.series.Series, (17520,), dtype('float64'))

## Anatomy

Four attributes matter: `.values` / `.to_numpy()` (the data), `.index` (labels), `.name`
(becomes the column name when you put it back in a DataFrame), `.dtype`.

In [2]:
print("name :", cons.name)
print("dtype:", cons.dtype)
print("index:", type(cons.index).__name__, cons.index[:3].tolist())
print("array:", cons.to_numpy()[:3], type(cons.to_numpy()))
cons.head(3)

name : consumption_mwh
dtype: float64
index: DatetimeIndex [Timestamp('2022-01-01 00:00:00+0000', tz='UTC'), Timestamp('2022-01-01 01:00:00+0000', tz='UTC'), Timestamp('2022-01-01 02:00:00+0000', tz='UTC')]
array: [26858.4 26177.8 26229.4] <class 'numpy.ndarray'>


time
2022-01-01 00:00:00+00:00    26858.4
2022-01-01 01:00:00+00:00    26177.8
2022-01-01 02:00:00+00:00    26229.4
Name: consumption_mwh, dtype: float64

## Creating a Series

From a list (default `RangeIndex`), from a dict (keys become the index), from a scalar
broadcast over an index, or by pulling a column out of a DataFrame.

**Pitfall:** a list of Python ints with one `None` becomes `float64` (NaN needs floats);
a list mixing strings and numbers becomes `object`, and `.mean()` will then fail.

In [3]:
s_list = pd.Series([10, 20, 30])
s_dict = pd.Series({"nuclear": 8.0, "ccgt": 12.5, "wind": 6.2}, name="capacity_gw")
s_scalar = pd.Series(0.0, index=["a", "b", "c"])
s_none = pd.Series([1, 2, None])
s_mixed = pd.Series([1, "2", 3])

for nm, s in [("list", s_list), ("dict", s_dict), ("scalar", s_scalar), ("None", s_none), ("mixed", s_mixed)]:
    print(f"{nm:7s} dtype={str(s.dtype):8s} index={type(s.index).__name__}")
s_dict

list    dtype=int64    index=RangeIndex
dict    dtype=float64  index=Index
scalar  dtype=float64  index=Index
None    dtype=float64  index=RangeIndex
mixed   dtype=object   index=RangeIndex


nuclear     8.0
ccgt       12.5
wind        6.2
Name: capacity_gw, dtype: float64

## Selecting values: labels vs positions

`s.loc[label]` selects by index label, `s.iloc[pos]` by position. Plain `s[...]` guesses,
which is fine for a string or datetime index but ambiguous for integer indexes.

**Pitfall:** `s[0]` on a non-integer index (e.g. dates) is deprecated in pandas 2.x and
will become label lookup. Always write `s.iloc[0]` when you mean "first element".

**Interview check:** "Is a label slice inclusive?" Yes for `.loc` (both ends), no for
`.iloc` (Python semantics).

In [4]:
print("iloc[0]      :", cons.iloc[0])
print("loc[label]   :", cons.loc["2022-01-01 00:00:00+00:00"])
print("loc slice    :", cons.loc["2022-01-01 00:00":"2022-01-01 02:00"].tolist())   # inclusive
print("iloc slice   :", cons.iloc[0:3].tolist())                                     # exclusive
print("partial str  :", cons.loc["2022-01-01"].shape)                                # whole day
print("at / iat     :", cons.at[cons.index[5]], cons.iat[5])

iloc[0]      : 26858.4
loc[label]   : 26858.4
loc slice    : [26858.4, 26177.8, 26229.4]
iloc slice   : [26858.4, 26177.8, 26229.4]
partial str  : (24,)
at / iat     : 26453.7 26453.7


### Boolean masks, `isin`, `between`, `nlargest`

Masks are Series of booleans aligned on the same index. Combine with `&`, `|`, `~`
and always parenthesise each comparison.

In [5]:
high = cons[(cons > 38_000) & (cons.index.hour == 18)]
print("hours above 38 GWh at 18:00:", len(high))

peak_hours = cons[cons.index.hour.isin([17, 18, 19])]
print("peak-hour rows:", len(peak_hours))

mid = cons[cons.between(29_000, 30_000)]
print("between 29k and 30k:", len(mid))

cons.nlargest(3)

hours above 38 GWh at 18:00: 96
peak-hour rows: 2190
between 29k and 30k: 1732


time
2022-02-07 18:00:00+00:00    40824.9
2023-01-10 17:00:00+00:00    40485.4
2022-01-19 18:00:00+00:00    40453.7
Name: consumption_mwh, dtype: float64

## Index alignment — the most important Series concept

Arithmetic between two Series aligns **on labels, not positions**. Labels present in only
one Series give NaN. This is what makes `df["a"] + df["b"]` safe, and what silently breaks
when two Series have different indexes.

In [6]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])
print(a + b)                      # union of labels, NaN where one side is missing
print()
print(a.add(b, fill_value=0))     # treat missing as 0

w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64

w    30.0
x     1.0
y    12.0
z    23.0
dtype: float64


**Pitfall:** `.values` throws the index away, so `a.values + b.values` adds by position.
Equally dangerous: assigning a NumPy array back into a DataFrame column is positional.

**Interview check:** "You computed a lag on a sorted copy and assigned it back to the
unsorted frame. Is it right?" Yes if you assign the *Series* (alignment by index rescues
you); no if you assigned `.values`.

In [7]:
daily = cons.resample("D").mean()
shuffled = daily.sample(frac=1, random_state=0)          # same data, different order

sorted_lag = daily.shift(1)                              # lag computed on the sorted series
aligned = shuffled.to_frame("cons").assign(lag_series=sorted_lag)          # aligned by index -> correct
positional = shuffled.to_frame("cons").assign(lag_values=sorted_lag.values) # positional -> wrong

check = aligned.join(positional["lag_values"]).sort_index()
check["series_ok"] = np.isclose(check["lag_series"], daily.shift(1), equal_nan=True)
check["values_ok"] = np.isclose(check["lag_values"], daily.shift(1), equal_nan=True)
check[["series_ok", "values_ok"]].mean()

series_ok    1.0
values_ok    0.0
dtype: float64

### Comparisons need identical labels

`a == b` between Series with different indexes raises rather than aligning. Use
`a.eq(b)` (aligns) or reindex first.

In [8]:
try:
    a == b
except ValueError as e:
    print("ValueError:", e)
print(a.eq(b))              # aligned comparison, NaN-labels compare False

ValueError: Can only compare identically-labeled Series objects
w    False
x    False
y    False
z    False
dtype: bool


## Missing values

`isna` / `notna` for detection, `fillna` / `interpolate` / `dropna` for handling,
`where(cond, other)` keeps values where the condition is True and replaces the rest;
`mask` is the inverse.

**Pitfall:** `s == np.nan` is always False. Use `s.isna()`.

In [9]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
temp = raw["temp_c"]
print("NaN count       :", temp.isna().sum())
print("== np.nan count :", (temp == np.nan).sum())
print("sentinel -999   :", (temp == -999).sum())

temp_clean = temp.mask(temp <= -100)                     # sentinel -> NaN
print("after mask      :", temp_clean.isna().sum())
print("interpolate     :", temp_clean.interpolate(limit=3).isna().sum())
print("where positive  :", temp_clean.where(temp_clean > 0).notna().mean().round(3))

NaN count       : 149
== np.nan count : 0
sentinel -999   : 63
after mask      : 212
interpolate     : 0
where positive  : 0.924


## Vectorised statistics and shape-preserving methods

Reductions return a scalar (`mean`, `std` with `ddof=1`, `quantile`, `idxmax`).
Shape-preserving methods return a Series with the same index (`rank`, `clip`, `cumsum`,
`diff`, `pct_change`, `shift`, `rolling`).

**Pitfall:** pandas `std()` uses `ddof=1`; NumPy `np.std` uses `ddof=0`.

In [10]:
print("mean     :", round(cons.mean(), 1))
print("std pandas (ddof=1):", round(cons.std(), 2), " numpy (ddof=0):", round(np.std(cons.to_numpy()), 2))
print("quantiles:", cons.quantile([0.05, 0.5, 0.95]).round(0).tolist())
print("idxmax   :", cons.idxmax(), "->", cons.max())
print("argmax   :", cons.argmax(), "(position)")
cons.describe().round(1)

mean     : 29315.3
std pandas (ddof=1): 4204.94  numpy (ddof=0): 4204.82
quantiles: [21824.0, 29672.0, 35870.0]
idxmax   : 2022-02-07 18:00:00+00:00 -> 40824.9
argmax   : 906 (position)


count    17520.0
mean     29315.3
std       4204.9
min      18092.9
25%      26447.1
50%      29672.0
75%      32370.1
max      40824.9
Name: consumption_mwh, dtype: float64

In [11]:
day = cons.loc["2023-01-24"]                # one cold day, 24 values
pd.DataFrame({
    "value": day,
    "rank": day.rank(),
    "clip_30k": day.clip(upper=30_000),
    "cumsum": day.cumsum(),
    "diff": day.diff(),
    "pct_change": day.pct_change().round(4),
    "shift1": day.shift(1),
    "roll3_shifted": day.shift(1).rolling(3).mean(),
}).head(6)

,value,rank,clip_30k,cumsum,diff,pct_change,shift1,roll3_shifted
time,,,,,,,,
2023-01-24 00:00:00+00:00,31399.6,7.0,30000.0,31399.6,NaN,NaN,NaN,NaN
2023-01-24 01:00:00+00:00,29679.8,5.0,29679.8,61079.4,-1719.8,-0.0548,31399.6,NaN
2023-01-24 02:00:00+00:00,29344.0,4.0,29344.0,90423.4,-335.8,-0.0113,29679.8,NaN
2023-01-24 03:00:00+00:00,29118.6,3.0,29118.6,119542.0,-225.4,-0.0077,29344.0,30141.133333
2023-01-24 04:00:00+00:00,28228.2,1.0,28228.2,147770.2,-890.4,-0.0306,29118.6,29380.800000
2023-01-24 05:00:00+00:00,29088.6,2.0,29088.6,176858.8,860.4,0.0305,28228.2,28896.933333


### `value_counts`, `unique`, `nunique`

`value_counts` is the fastest way to see a distribution. `normalize=True` gives shares,
`dropna=False` keeps NaN as its own row, `bins=` bins a numeric Series.

In [12]:
tariff = pd.read_csv("../data/meters.csv")["tariff"]
print(tariff.value_counts(dropna=False))
print()
print(tariff.value_counts(normalize=True).round(3))
print()
print("unique:", tariff.unique(), " nunique:", tariff.nunique(), " (dropna=False:", tariff.nunique(dropna=False), ")")
cons.value_counts(bins=5).sort_index()

tariff
Fixed       143
Variable     94
TOU          50
NaN          13
Name: count, dtype: int64

tariff
Fixed       0.498
Variable    0.328
TOU         0.174
Name: proportion, dtype: float64

unique: ['Fixed' 'Variable' 'TOU' nan]  nunique: 3  (dropna=False: 4 )


(18070.167, 22639.3]    1365
(22639.3, 27185.7]      3838
(27185.7, 31732.1]      7094
(31732.1, 36278.5]      4538
(36278.5, 40824.9]       685
Name: count, dtype: int64

## `map`, `apply`, `replace`, `astype`

- `map(dict_or_func)` element-wise, unmapped keys become NaN.
- `apply(func)` element-wise for a Series (slow, Python loop; prefer vectorised).
- `replace({old: new})` substitutes values, leaves others untouched.
- `astype` converts dtype; `pd.to_numeric(errors="coerce")` for dirty text.

**Pitfall:** `map` with a dict returns NaN for every value not in the dict; `replace`
keeps them. People mix these up and lose data.

In [13]:
region = pd.read_csv("../data/meters.csv")["region"]
lookup = {"London": "South", "Wales": "West", "Scotland": "North"}
print("map     -> NaN for unmapped:", region.map(lookup).isna().sum())
print("replace -> unmapped kept   :", region.replace(lookup).isna().sum())
print(region.replace(lookup).value_counts().head(8))

map     -> NaN for unmapped: 115
replace -> unmapped kept   : 0
region
North       122
South        96
Midlands     44
West         32
london        3
wales         1
north         1
midlands      1
Name: count, dtype: int64


In [14]:
price_txt = raw["price_eur_mwh"]                      # object dtype with "missing" strings
print(price_txt.dtype, "| sample:", price_txt.head(3).tolist())
try:
    price_txt.astype(float)
except ValueError as e:
    print("astype(float) fails:", str(e)[:60])
price = pd.to_numeric(price_txt, errors="coerce")
print("coerced NaN:", price.isna().sum(), "| dtype:", price.dtype)

%timeit price.apply(lambda v: v * 1.1)
%timeit price * 1.1

object | sample: ['70.16', '23.19', '115.79']
astype(float) fails: could not convert string to float: 'missing'
coerced NaN: 100 | dtype: float64


2.19 ms ± 15.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


49.4 µs ± 278 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Accessors: `.str`, `.dt`, `.cat`

Vectorised methods live behind accessors: `.str` for strings, `.dt` for datetimes,
`.cat` for categoricals. They only exist when the dtype matches, which is a useful
sanity check: `AttributeError: Can only use .dt accessor` means your dates are still
strings.

In [15]:
reg = pd.read_csv("../data/meters.csv")["region"]
print(reg.str.lower().value_counts().head(3).to_dict())
print("contains 'lon' (case-insens):", reg.str.contains("lon", case=False).sum())
print("lengths:", reg.str.len().describe()[["min", "max"]].to_dict())

t = df["time"]
print("dt.hour:", t.dt.hour.iloc[:3].tolist(), "| dt.dayofweek:", t.dt.dayofweek.iloc[:3].tolist(), "| dt.tz:", t.dt.tz)

try:
    raw["time"].dt.hour
except AttributeError as e:
    print("strings have no .dt:", str(e)[:50])

reg_cat = reg.str.title().astype("category")
print("categories:", reg_cat.cat.categories.tolist(), "| codes:", reg_cat.cat.codes.iloc[:5].tolist())
print("memory object vs category (bytes):", reg.memory_usage(deep=True), reg_cat.memory_usage(deep=True))

{'london': 99, 'north': 66, 'scotland': 57}
contains 'lon' (case-insens): 99
lengths: {'min': 5.0, 'max': 8.0}
dt.hour: [0, 1, 2] | dt.dayofweek: [5, 5, 5] | dt.tz: UTC
strings have no .dt: Can only use .dt accessor with datetimelike values
categories: ['London', 'Midlands', 'North', 'Scotland', 'Wales'] | codes: [0, 0, 0, 3, 1]
memory object vs category (bytes): 19133 917


## Index manipulation

`reindex` conforms a Series to a new index (introducing NaN, optionally filling);
`rename` relabels; `reset_index` turns the index into a column and returns a DataFrame;
`sort_index` / `sort_values` order it. Check `index.is_unique` and
`index.is_monotonic_increasing` before any shift or rolling.

In [16]:
full_idx = pd.date_range(cons.index.min(), cons.index.max(), freq="h", tz="UTC")
raw_cons = (raw.assign(time=pd.to_datetime(raw["time"], utc=True))
               .drop_duplicates("time").set_index("time")["consumption_mwh"].sort_index())
print("raw unique/monotonic:", raw_cons.index.is_unique, raw_cons.index.is_monotonic_increasing)
print("raw rows:", len(raw_cons), "| full grid:", len(full_idx))
gaps = raw_cons.reindex(full_idx)
print("missing hours after reindex:", gaps.isna().sum())

s_dict.rename({"ccgt": "gas"}).rename("capacity")            # relabel index and name

raw unique/monotonic: True True
raw rows: 17442 | full grid: 17520
missing hours after reindex: 78


nuclear     8.0
gas        12.5
wind        6.2
Name: capacity, dtype: float64

In [17]:
as_frame = cons.reset_index()               # Series -> DataFrame with 'time' and 'consumption_mwh'
print(type(as_frame).__name__, as_frame.columns.tolist())
print(cons.sort_values(ascending=False).head(3))

DataFrame ['time', 'consumption_mwh']
time
2022-02-07 18:00:00+00:00    40824.9
2023-01-10 17:00:00+00:00    40485.4
2022-01-19 18:00:00+00:00    40453.7
Name: consumption_mwh, dtype: float64


### Duplicate labels in the index

With duplicated labels, `s[label]` returns a *Series*, `reindex` raises, and alignment
multiplies rows. Detect with `index.duplicated()`.

**Interview check:** "Why did my reindex to the hourly grid fail?" Duplicated timestamps.

In [18]:
dup_cons = (raw.assign(time=pd.to_datetime(raw["time"], utc=True))
               .set_index("time")["consumption_mwh"].sort_index())
dups = dup_cons.index[dup_cons.index.duplicated()]
print("duplicated labels:", len(dups))
print("s[label] with a duplicated label returns:", type(dup_cons[dups[0]]).__name__, "of length", len(dup_cons[dups[0]]))
try:
    dup_cons.reindex(full_idx)
except ValueError as e:
    print("reindex fails:", e)
print("fix: ", dup_cons[~dup_cons.index.duplicated(keep='last')].reindex(full_idx).isna().sum(), "missing after dedupe")

duplicated labels: 15
s[label] with a duplicated label returns: Series of length 2
reindex fails: cannot reindex on an axis with duplicate labels
fix:  78 missing after dedupe


## Combining Series

- `pd.concat([s1, s2])` stacks (axis=0) or puts them side by side as columns (axis=1).
- `s1.combine_first(s2)` fills NaN in s1 from s2 (patching from a second source).
- `s1.update(s2)` overwrites in place where s2 has values.
- `s.to_frame(name)` converts to a one-column DataFrame.

In [19]:
primary = cons.loc["2022-03-26":"2022-03-28"].copy()
primary.iloc[10:20] = np.nan                           # pretend a feed dropped out
backup = (cons.loc["2022-03-26":"2022-03-28"] * 1.001)  # a second source, slightly different

patched = primary.combine_first(backup)
print("primary NaN:", primary.isna().sum(), "| patched NaN:", patched.isna().sum())

side_by_side = pd.concat([primary.rename("primary"), backup.rename("backup")], axis=1)
print(side_by_side.iloc[9:12].round(1))

stacked = pd.concat([s_dict, pd.Series({"solar": 3.1}, name="capacity_gw")])
print(stacked)

primary NaN: 10 | patched NaN: 0
                           primary   backup
time                                       
2022-03-26 09:00:00+00:00  31849.3  31881.1
2022-03-26 10:00:00+00:00      NaN  31361.8
2022-03-26 11:00:00+00:00      NaN  30135.9
nuclear     8.0
ccgt       12.5
wind        6.2
solar       3.1
Name: capacity_gw, dtype: float64


## MultiIndex Series

A groupby on two keys returns a Series with a two-level index. Select with tuples, drop
a level with `xs`, or `unstack` a level into columns to get a table.

In [20]:
profile = cons.groupby([cons.index.dayofweek.rename("dow"), cons.index.hour.rename("hour")]).mean()
print(type(profile.index).__name__, profile.index.names, profile.shape)
print("one cell     :", round(profile.loc[(0, 18)], 1))
print("Monday hours :", profile.xs(0, level="dow").round(0).iloc[[0, 8, 18]].to_dict())
profile.unstack("hour").round(0).iloc[:, [0, 6, 12, 18, 23]]

MultiIndex ['dow', 'hour'] (168,)
one cell     : 36061.4
Monday hours : {0: 26013.0, 8: 32338.0, 18: 36061.0}


hour,0,6,12,18,23
dow,,,,,
0,26013.0,27883.0,31672.0,36061.0,27007.0
1,26019.0,27759.0,31377.0,35814.0,27049.0
2,26047.0,27737.0,31431.0,36026.0,27016.0
3,26097.0,27937.0,31438.0,36027.0,27019.0
4,25913.0,27793.0,31497.0,35910.0,26981.0
5,23937.0,25798.0,29366.0,33889.0,24997.0
6,24014.0,25730.0,29374.0,33839.0,24902.0


## Conversions

`to_numpy()`, `to_list()`, `to_dict()`, `to_frame()`, `item()` for a length-1 Series,
`squeeze()` to turn a 1-column DataFrame into a Series. `df["col"]` gives a Series,
`df[["col"]]` gives a DataFrame.

In [21]:
print(type(df["temp_c"]).__name__, "vs", type(df[["temp_c"]]).__name__)
print(df[["temp_c"]].squeeze().shape)
print(s_dict.to_dict())
print(cons.iloc[[0]].item())                 # .item() only works on a length-1 Series
print(cons.head(2).to_frame().columns.tolist())

Series vs DataFrame
(17520,)
{'nuclear': 8.0, 'ccgt': 12.5, 'wind': 6.2}
26858.4
['consumption_mwh']


## Pitfall catalogue

Each cell below shows a mistake that runs (or fails) and the fix.

In [22]:
# 1. Truth value of a Series is ambiguous
s = pd.Series([1, 0, 3])
try:
    if s:
        pass
except ValueError as e:
    print("if s:            ->", str(e)[:60])
print("use s.any()/all():", s.any(), s.all(), "| .empty:", s.empty)

if s:            -> The truth value of a Series is ambiguous. Use a.empty, a.boo
use s.any()/all(): True False | .empty: False


In [23]:
# 2. `in` tests the INDEX, not the values
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print("10 in s     :", 10 in s)              # False: looks at labels
print("'a' in s    :", "a" in s)
print("10 in s.values:", 10 in s.values, "| s.isin([10]).any():", s.isin([10]).any())

10 in s     : False
'a' in s    : True
10 in s.values: True | s.isin([10]).any(): True


In [24]:
# 3. Chained assignment: writing into a Series pulled out of a filtered frame
import warnings
before = df["temp_c"].min()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                 # pandas may or may not warn depending on version/mode
    df[df["temp_c"] < -5]["temp_c"] = 0.0           # writes into a temporary copy
print("min before:", before, "| min after:", df["temp_c"].min(), "-> original untouched (silent no-op)")

sub_col = df[df["temp_c"] < -5]["temp_c"]           # a Series that may be a copy: mutating it is unreliable
print("fix: df.loc[mask, 'temp_c'] = value, or sub = df[mask].copy() before editing")

min before: -6.4 | min after: -6.4 -> original untouched (silent no-op)
fix: df.loc[mask, 'temp_c'] = value, or sub = df[mask].copy() before editing


In [25]:
# 4. Integer Series becomes float when NaN appears; bool becomes object
ints = pd.Series([1, 2, 3])
print(ints.dtype, "->", ints.reindex([0, 1, 2, 3]).dtype)
bools = pd.Series([True, False])
print(bools.dtype, "->", bools.reindex([0, 1, 2]).dtype)
print("nullable alternative:", ints.astype("Int64").reindex([0, 1, 2, 3]).dtype)

int64 -> float64
bool -> object
nullable alternative: Int64


In [26]:
# 5. Sum of all-NaN is 0, mean is NaN; min_count changes it
allnan = pd.Series([np.nan, np.nan])
print("sum:", allnan.sum(), "| sum(min_count=1):", allnan.sum(min_count=1), "| mean:", allnan.mean())

sum: 0.0 | sum(min_count=1): nan | mean: nan


In [27]:
# 6. NumPy ufuncs keep the index; NumPy reductions do not
print(type(np.log(cons.head(3))).__name__, np.log(cons.head(3)).index[0])
print(type(np.mean(cons)).__name__)

Series 2022-01-01 00:00:00+00:00
float64


In [28]:
# 7. Empty Series defaults to object dtype, and shift(-1) plus dropna misaligns X and y
print("empty dtype:", pd.Series(dtype="float64").dtype, "vs", pd.Series([]).dtype)

y = cons.shift(-1)                     # next hour
X = cons.to_frame("lag0")
Xd, yd = X.dropna(), y.dropna()        # different lengths -> misaligned if you use .values
print("len X:", len(Xd), "len y:", len(yd), "-> build one frame and dropna once:")
frame = pd.concat([X, y.rename("target")], axis=1).dropna()
print("aligned rows:", len(frame))

empty dtype: float64 vs object
len X: 17520 len y: 17519 -> build one frame and dropna once:
aligned rows: 17519


## Quick reference

| Task | Series idiom |
|---|---|
| first / last element | `s.iloc[0]`, `s.iloc[-1]` |
| by label, inclusive slice | `s.loc[a:b]` |
| mask | `s[(s > x) & (s < y)]`, `s.between(x, y)` |
| aligned arithmetic with fill | `s1.add(s2, fill_value=0)` |
| aligned comparison | `s1.eq(s2)` not `s1 == s2` |
| replace sentinels | `s.mask(s <= -100)` |
| distribution | `s.value_counts(normalize=True, dropna=False)` |
| lookup table | `s.map(dict)` (unmapped -> NaN) vs `s.replace(dict)` |
| dirty numbers | `pd.to_numeric(s, errors="coerce")` |
| conform to a grid | `s.reindex(pd.date_range(...))` after dedupe |
| lagged rolling feature | `s.shift(1).rolling(24).mean()` |
| patch from second source | `s1.combine_first(s2)` |
| two-key summary as table | `s.groupby([k1, k2]).mean().unstack()` |
| Series -> DataFrame | `s.to_frame("name")`, `s.reset_index()` |
| check before shift/rolling | `s.index.is_unique`, `s.index.is_monotonic_increasing` |